# T25 / E06 — Định vị chú ý trên ISE-DSC01

**Thí nghiệm quyết định CH1.**

Mọi kết quả đến giờ suy ra cơ chế từ **điểm phân loại**: đặc trưng chunk-aware giúp ích, vậy chú
ý hẳn phải rơi vào chỗ có nghĩa. Đó là suy luận gián tiếp. Một bộ phát hiện có thể ăn điểm trên
ViHallu nhờ bất kỳ thứ gì tương quan với ảo giác.

Thí nghiệm này đọc **thẳng**: đoạn được chú ý nhiều nhất **có phải** đoạn chứa bằng chứng vàng
không. ISE-DSC01 có bằng chứng nguyên văn cho nhãn SUPPORTED và REFUTED, nên câu hỏi trả lời
được mà không cần bộ phân loại nào.

## Hai câu hỏi, trên hai nửa rời nhau của cùng tập dev

| Nửa | Nhãn | Số mẫu (dev) | Đo gì |
|---|---|---|---|
| Có bằng chứng | `no` + `intrinsic` | 2.377 | hit@1, hit@3, MRR so với sàn ngẫu nhiên |
| Không bằng chứng | `extrinsic` (NEI) | 1.269 | entropy có cao hơn không, kèm kiểm định |

## Đã đo trên CPU trước khi đặt lịch GPU

400 mẫu train, bằng chính bộ chia đoạn sẽ dùng:

```
  ngữ cảnh          794 token trung bình, p95 1.671, lớn nhất 2.313
  vượt trần 4.096   0,0 %
  số đoạn           21,7 trung bình, trung vị 19, lớn nhất 63
  định vị bằng chứng 400/400
  sàn ngẫu nhiên    1/n trung bình = 0,0614
  vị trí đoạn vàng  trung bình 0,504 — RẢI ĐỀU
```

Bốn điều rút ra, và điều cuối là quan trọng nhất:

1. **Không mẫu nào bị cắt ngữ cảnh.** Trái với dự đoán trong `CLAUDE.md`, ISE-DSC01 sau chuẩn hóa
   không dài tới mức chạm trần 4.096 token. Đường cắt ngắn vẫn chưa được thử thật.
2. **21,7 đoạn mỗi ngữ cảnh**, gấp bốn ViHallu. Đây mới là chỗ chunk-aware có đất diễn.
3. **Sàn ngẫu nhiên chỉ 0,0614**, nên hit@1 rất dễ phân biệt với đoán mò — khác hẳn ViHallu nơi
   5,3 đoạn cho sàn 0,19.
4. **Đoạn vàng rải đều trong ngữ cảnh** (vị trí tương đối trung bình 0,504). Nếu nó dồn về đầu
   hay cuối thì một bộ đoán "luôn chọn đoạn đầu" đã thắng sàn, và hit@1 cao sẽ chẳng chứng minh
   được gì. Nó rải đều nên không có mẹo nào ăn được.

## Đọc kết quả thế nào — viết trước khi thấy số

- **hit@1 vượt sàn nhiều lần** → cơ chế được xác nhận trực tiếp. CH1 đứng.
- **hit@1 quanh sàn** → chú ý **không** rơi vào đoạn bằng chứng. Đây là kết quả **bác CH1**, và
  phải báo cáo đúng như vậy. Lúc đó phần chunk-aware ăn điểm ở E03 là nhờ thứ khác, và phải đi
  tìm thứ khác đó chứ không đi tìm cách đọc dễ chịu hơn.
- **Entropy nhãn NEI cao hơn với cỡ ảnh hưởng đáng kể** → nửa "ngoại lai thì chú ý tản" của giả
  thuyết được xác nhận trên dữ liệu có nhãn thật.

**Với cỡ mẫu này, `p` gần như chắc chắn rất nhỏ dù khác biệt có nhỏ đến đâu.** Script vì thế in
cỡ ảnh hưởng rank-biserial và bắt đọc nó trước. Đừng dẫn `p` làm bằng chứng chính.

## Chi phí

Ngữ cảnh dài gấp ba ViHallu nên mỗi mẫu tốn khoảng **1,1 giây**. Toàn bộ tập dev 3.646 mẫu
khoảng **70 phút**. Chỉ chạy tập dev: E06 không huấn luyện gì nên không cần train, và để dành
tập test cho E07 với E16.

## Chuẩn bị

Hai ô đầu giống notebook T24. Khoảng 2 phút.

In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

đã clone mới
/kaggle/working/vihallulens
commit: 4abf0be T25: công cụ E06 định vị chú ý, chờ chạy GPU (#60)


In [2]:
# Ô 2 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit.
# hình đọc 7B. Không có nó thì mô hình phải nạp ở float16 và tràn 16 GB.
!pip install -q --no-deps -e .
!pip install -q -U transformers accelerate bitsandbytes

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires pyvi, which is not installed.
vihallulens 0.1.0 requires rank-bm25, which is not install

In [3]:
# Ô 3 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, chạy CPU.
# test_localization.py và test_stats.py là bộ kiểm viết riêng cho T25. Chúng bắt được một lỗi
# thật lúc phát triển: cách xử lý hòa khi xếp hạng vốn lạc quan, sẽ thổi phồng hit@1.
get_ipython().system("python scripts/probe_env.py")
get_ipython().system("python scripts/normalize_data.py --dataset isedsc01")
get_ipython().system("python scripts/split_data.py --only isedsc01")
get_ipython().system(
    "python -m pytest tests/test_localization.py tests/test_stats.py"
    " tests/test_chunking_config.py -q"
)


MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : 4abf0be T25: công cụ E06 định vị chú ý, chờ chạy GPU (#60)
  python           : 3.12.13
  torch            : 2.10.0+cu128
  transformers     : 5.16.1
  bitsandbytes     : 0.50.2
  accelerate       : 1.14.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA ISEDSC01
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  số dòng               : 36,369
  ngữ cảnh duy nhất   

In [4]:
# Ô 4 — xác nhận tiền đề của thí nghiệm trên chính tập sẽ chạy. Khoảng 2 phút, CPU.
# Nếu đoạn vàng KHÔNG rải đều thì hit@1 cao sẽ không chứng minh được gì, vì một bộ đoán theo vị
# trí đã thắng sàn. Đây là chỗ phải kiểm trước, không phải sau.
import sys

sys.path.insert(0, "src")
import numpy as np
import pandas as pd

from vihallulens.data.chunking import chunk_context, locate_evidence_chunk

d = pd.read_parquet("data/interim/isedsc01_dev.parquet")
ev = d["evidence"].fillna("").str.strip()
sub = d[ev.str.len() > 0].assign(_ev=ev[ev.str.len() > 0])
print(f"tập dev: {len(d):,} mẫu, {len(sub):,} có bằng chứng")

pos, nch, miss = [], [], 0
for _, r in sub.iterrows():
    ch = chunk_context(r["context"], strategy="sentence", min_words=5)
    g = locate_evidence_chunk(ch, r["_ev"], r["context"])
    nch.append(len(ch))
    if g is None:
        miss += 1
    else:
        pos.append(g / max(len(ch) - 1, 1))

nch = np.array(nch)
pos = np.array(pos)
print(f"số đoạn        : TB {nch.mean():.1f}, trung vị {np.median(nch):.0f}, "
      f"lớn nhất {nch.max()}")
print(f"sàn ngẫu nhiên : {np.mean(1 / nch):.4f}")
print(f"không định vị  : {miss}/{len(sub)}")
print(f"vị trí đoạn vàng: TB {pos.mean():.3f}, trung vị {np.median(pos):.3f}, "
      f"phần ở nửa đầu {np.mean(pos < 0.5):.1%}")
assert 0.35 < pos.mean() < 0.65, "đoạn vàng lệch về một phía — hit@1 sẽ không diễn giải được"
print("\nĐạt: đoạn vàng rải đều, không có mẹo vị trí nào ăn được sàn.")

tập dev: 3,646 mẫu, 2,377 có bằng chứng
số đoạn        : TB 22.7, trung vị 19, lớn nhất 161
sàn ngẫu nhiên : 0.0614
không định vị  : 0/2377
vị trí đoạn vàng: TB 0.519, trung vị 0.524, phần ở nửa đầu 45.6%

Đạt: đoạn vàng rải đều, không có mẹo vị trí nào ăn được sàn.


## Trích đặc trưng

Một ô, khoảng **70 phút**. Chạy lại được: mẫu nào xong ghi xuống ngay.

**Đọc gì trong lúc chạy:** dòng `mẫu có bằng chứng` phải báo khoảng 2.377/3.646 và nói thêm
`→ ghi thêm gold_rank cho E06`. Không thấy dòng ấy nghĩa là cột `evidence` không tới được bộ
trích, và lượt chạy sẽ không đo được gì.

In [5]:
# Ô 5 — trích đặc trưng tập dev ISE-DSC01. Khoảng 70 phút.
!python scripts/extract_features.py --config configs/e06_localization_isedsc01.yaml --split dev


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e06_localization_isedsc01.yaml  (hash 15ef31521fd6)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : isedsc01/dev, 3,646 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 2,377/3,646  → ghi thêm gold_rank cho E06
  đã có sẵn             : 0 mẫu trong isedsc01_dev_15ef31521fd6.jsonl
  còn phải chạy         : 3,646 mẫu
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 3.57MB/s]
tokenizer_config.json: 7.30kB [00:00, 26.8MB/s]
vocab.json: 2.78MB [00:00, 48.8MB/s]
merges.txt: 1.67MB [00:00, 113MB/s]
tokenizer.json: 7.03MB [00:00, 136MB/s]
model.safetensors.index.json: 27.8kB [00:00, 73.6MB/s]
Fetching 4 files: 100%|██████████████████

## Đo định vị

Chạy CPU, vài giây. Copy toàn bộ output.

In [6]:
# Ô 6 — đo hit@1, hit@3, MRR và kiểm định entropy.
!python scripts/run_localization.py --config configs/e06_localization_isedsc01.yaml --split dev


E06 — ĐỊNH VỊ CHÚ Ý TRÊN NGỮ CẢNH DÀI
  cấu hình              : configs/e06_localization_isedsc01.yaml  (trích 15ef31521fd6)
  bộ dữ liệu            : isedsc01/dev, 3,646 mẫu
  chia đoạn             : sentence
  lưới lớp × đầu        : 27 × 28
  có bằng chứng định vị được : 2,376
  nhãn không bằng chứng      : 1,269
  bị cắt ngữ cảnh            : 6/3,646
  không định vị được bằng chứng : 1  ← loại khỏi phần định vị

--------------------------------------------------------------------------------
ĐỊNH VỊ CHÚ Ý — 2,376 mẫu có bằng chứng, trung bình 22.6 đoạn mỗi ngữ cảnh
--------------------------------------------------------------------------------
  Chỉ số      đầu tốt nhất  sàn ngẫu nhiên   gấp sàn    trung bình mọi đầu
  hit@1             0.8779          0.0614    14.29x                0.3320   (lớp 14, đầu 6)
  hit@3             0.9562          0.1843     5.19x                0.5177   (lớp 16, đầu 7)
  mrr               0.9200          0.1987     4.63x                0.4709   (lớp

In [7]:
# Ô 7 — kiểm toàn vẹn shard trước khi rời phiên. Vài giây, CPU.
import json
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config

run = extraction_hash(load_config("configs/e06_localization_isedsc01.yaml"))
path = Path("data/processed") / f"isedsc01_dev_{run}.jsonl"
if not path.exists():
    raise SystemExit(f"THIEU {path.name} - chay lai o trich truoc khi roi phien.")
rows = [json.loads(line) for line in path.open(encoding="utf-8") if line.strip()]
ids = {r["sample_id"] for r in rows}
blocks = [k for k in rows[0] if k.startswith(("lookback_", "chunk_", "top1_"))]
gold = [r for r in rows if "gold_rank" in r]
widths = {len(r["gold_rank"]) for r in gold}
ok = len(rows) == 3646 and len(ids) == len(rows) and len(blocks) == 7 and widths == {756}
print(f"  {path.name}")
print(f"  {len(rows):,} dòng, {len(ids):,} id, {len(blocks)} khối đặc trưng")
print(f"  {len(gold):,} mẫu có gold_rank, độ rộng {widths or 'không có'}")
print(f"  bị cắt ngữ cảnh: {sum(r['truncated'] for r in rows):,}")
print("\nShard hợp lệ." if ok else "\nCÓ VẤN ĐỀ — chạy lại ô trích trước khi rời phiên.")

  isedsc01_dev_15ef31521fd6.jsonl
  3,646 dòng, 3,646 id, 7 khối đặc trưng
  2,376 mẫu có gold_rank, độ rộng {756}
  bị cắt ngữ cảnh: 6

Shard hợp lệ.


In [8]:
# Ô 8 — lấy kết quả về.
import shutil
from pathlib import Path

shutil.copy("results/runs.jsonl", "/kaggle/working/runs.jsonl")
print("runs.jsonl")
for path in sorted(Path("data/processed").glob("isedsc01_*.jsonl")):
    shutil.copy(path, f"/kaggle/working/{path.name}")
    print(f"{path.name}  {path.stat().st_size / 1024**2:.1f} MB")

runs.jsonl
isedsc01_dev_15ef31521fd6.jsonl  189.2 MB


## Sau khi chạy

Dán toàn bộ output của ô 6. Ba dòng quyết định là `hit@1`, `hit@3`, `mrr` cùng cột `gấp sàn`, và
cả khối `ĐỘ TẢN CỦA CHÚ Ý`.

Rồi Quick Save, tải notebook về, chép đè lên `notebooks/t25_dinh_vi_chu_y_t4.ipynb`. **Đừng dùng
Save & Run All** — nó chạy lại 70 phút GPU không để làm gì.

Tải cả `runs.jsonl` và shard `isedsc01_dev_*.jsonl` (khoảng 400 MB) về, vì T26 dùng lại chính
lượt trích này cho E07 và sẽ không phải chạy GPU nữa.

## Một chỗ cố ý không làm

Bảng 2 trong `docs/EXPERIMENTS.md` có sẵn dòng "Cửa sổ 128 token" bên cạnh "Chia theo câu". Kế
hoạch ấy viết **trước** khi T24 chốt cách chia. Giờ chia theo câu đã là cấu hình chốt cho mọi
thí nghiệm còn lại, nên chạy thêm một lượt trích 70 phút cho cách chia đã bị loại là tiêu hạn
mức GPU để điền một ô không ai dùng. Nếu hội đồng hỏi, câu trả lời nằm ở Bảng 3.